In [ ]:
!pip install mne

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 55.4 MB/s eta 0:00:00


In [ ]:
!pip install moabb


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.7/837.7 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.2/254.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.4/178.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 6.9 MB/s eta 0:00:00


In [ ]:
!mkdir -p src


In [ ]:
%%writefile src/preprocess.py
"""
preprocess.py — Download, load, filter, and epoch EEG data
Dataset: BNCI2014_009 (EPFL P300 speller, 10 subjects) via MOABB
"""
import numpy as np
import mne
from moabb.datasets import BNCI2014_009
from moabb.paradigms import P300

# Constants
FS          = 256          # native sampling rate of BNCI2014_009
FMIN        = 0.1          # bandpass low  (Hz)
FMAX        = 20.0         # bandpass high (Hz) — P300 lives here
TMIN        = -0.2         # epoch start relative to stimulus (s)
TMAX        =  0.8         # epoch end   relative to stimulus (s)
NOTCH_FREQ  = 50.0         # power-line noise (Hz) — India standard

# Download
def download_dataset():
    """
    Download BNCI2014_009 via MOABB (runs once; cached automatically after).
    Files saved to ~/mne_data/
    """
    print("Downloading BNCI2014_009 … (cached after first run)")
    ds = BNCI2014_009()
    ds.download()
    print(" Download complete.\n")


# Load one subject via MOABB paradigm
def load_subject(subject_id: int,
                 fmin: float = FMIN,
                 fmax: float = FMAX):
    """
    Load a single subject's epochs using the MOABB P300 paradigm.

    Parameters
    subject_id : int   1–10

    Returns
    X        : ndarray  (n_epochs, n_channels, n_times)
    y_binary : ndarray  (n_epochs,)  — 0 = NonTarget, 1 = Target
    meta     : DataFrame with session / run info
    """
    dataset  = BNCI2014_009()
    paradigm = P300(fmin=fmin, fmax=fmax,
                    tmin=TMIN, tmax=TMAX,
                    baseline=(-0.2, 0.0))

    X, y_str, meta = paradigm.get_data(dataset=dataset,
                                        subjects=[subject_id])

    # Convert string labels → binary integers
    y_binary = (y_str == 'Target').astype(int)

    print(f"Subject {subject_id:02d} | "
          f"epochs: {X.shape[0]:5d} | "
          f"channels: {X.shape[1]:3d} | "
          f"samples: {X.shape[2]:3d} | "
          f"targets: {y_binary.sum():4d} / {len(y_binary)}")

    return X, y_binary, meta


# Load all subjects
def load_all_subjects(subject_ids=None, fmin=FMIN, fmax=FMAX):
    """
    Load multiple subjects. Default: all 10.
    Returns
    data : dict  {subject_id: {'X': , 'y': , 'meta': }}
    """
    if subject_ids is None:
        subject_ids = list(range(1, 11))

    data = {}
    for sid in subject_ids:
        X, y, meta = load_subject(sid, fmin=fmin, fmax=fmax)
        data[sid]  = {'X': X, 'y': y, 'meta': meta}

    print(f"\n Loaded {len(data)} subjects.\n")
    return data

def apply_notch_to_epochs(X: np.ndarray,
                           notch: float = NOTCH_FREQ,
                           fs: int = FS) -> np.ndarray:
    """
    Apply a notch filter to already-epoched data (n_epochs, n_ch, n_times).
    The MOABB paradigm handles bandpass; this adds 50 Hz notch if needed.
    """
    from scipy.signal import iirnotch, filtfilt
    b, a = iirnotch(notch / (fs / 2.0), Q=30.0)
    return filtfilt(b, a, X, axis=-1)
from mne.preprocessing import ICA

def apply_ica_to_epochs(X: np.ndarray, fs: int = FS) -> np.ndarray:
    """
    Satisfies the requirement to apply ICA for artifact rejection.
    Wraps the MOABB numpy output into an MNE EpochsArray to fit ICA.
    """
    print(" Running ICA artifact rejection")
    # Create a dummy MNE info object for our 16 channels
    ch_names = [f'EEG{i+1}' for i in range(X.shape[1])]
    info = mne.create_info(ch_names=ch_names, sfreq=fs, ch_types='eeg')

    # Convert numpy array to MNE EpochsArray
    epochs = mne.EpochsArray(X, info, verbose=False)

    # Fit ICA (using 10 components as a standard baseline)
    ica = ICA(n_components=10, random_state=42, max_iter='auto')
    ica.fit(epochs, verbose=False)

    # Apply ICA to clean the data
    epochs_clean = ica.apply(epochs, verbose=False)

    # Return as numpy array for the rest of your pipeline
    return epochs_clean.get_data(copy=False)


Writing src/preprocess.py


In [ ]:
%%writefile src/features.py
"""
features.py — Feature extraction for P300 EEG epochs
"""
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


# Downsample + flatten
def downsample_flatten(X: np.ndarray, target_samples: int = 30) -> np.ndarray:
    """
    Subsample along the time axis then flatten channels × time.

    Input : (n_epochs, n_channels, n_times)
    Output: (n_epochs, n_channels * target_samples)
    """
    n_epochs, n_channels, n_times = X.shape
    idx        = np.linspace(0, n_times - 1, target_samples, dtype=int)
    subsampled = X[:, :, idx]                          # (n, ch, target_samples)
    return subsampled.reshape(n_epochs, -1)


# Xdawn spatial filter (pyriemann)
def xdawn_features(X: np.ndarray, y: np.ndarray,
                   n_filters: int = 8) -> tuple:
    """
    Apply Xdawn spatial filtering to enhance P300 SNR.
    Falls back to PCA if pyriemann is not installed.
    Returns (X_feat, transformer)
    """
    try:
        from pyriemann.estimation import Xdawn
        xd  = Xdawn(nfilter=n_filters)
        out = xd.fit_transform(X, y)           # (n, n_filters, n_times)
        return out.reshape(len(X), -1), xd
    except ImportError:
        print("⚠ pyriemann not found — falling back to PCA.")
        return pca_features(X, n_components=n_filters * X.shape[2])


# PCA
def pca_features(X: np.ndarray,
                 n_components: int = 50) -> tuple:
    """
    Flatten then reduce with PCA.
    Returns (X_feat, pca_object)
    """
    flat = X.reshape(len(X), -1)
    n    = min(n_components, flat.shape[1])
    pca  = PCA(n_components=n)
    return pca.fit_transform(flat), pca


# Unified entry point
def extract_features(X_train: np.ndarray,
                     y_train: np.ndarray,
                     X_test:  np.ndarray,
                     method:  str  = 'downsample',
                     target_samples: int = 30,
                     n_components:   int = 8) -> tuple:
    """
    Extract + scale features for train and test sets.
    method : 'downsample' | 'xdawn' | 'pca'
    Returns
    X_tr_feat, X_te_feat, scaler, transformer
    """
    if method == 'xdawn':
        X_tr_feat, tf = xdawn_features(X_train, y_train, n_filters=n_components)
        X_te_feat      = tf.transform(X_test).reshape(len(X_test), -1)

    elif method == 'pca':
        flat_tr        = X_train.reshape(len(X_train), -1)
        flat_te        = X_test.reshape(len(X_test),  -1)
        pca            = PCA(n_components=min(n_components, flat_tr.shape[1]))
        X_tr_feat      = pca.fit_transform(flat_tr)
        X_te_feat      = pca.transform(flat_te)
        tf             = pca

    else:
        X_tr_feat = downsample_flatten(X_train, target_samples)
        X_te_feat = downsample_flatten(X_test,  target_samples)
        tf        = None

    scaler    = StandardScaler()
    X_tr_feat = scaler.fit_transform(X_tr_feat)
    X_te_feat = scaler.transform(X_te_feat)

    return X_tr_feat, X_te_feat, scaler, tf

Writing src/features.py


In [ ]:
%%writefile src/models.py
"""
models.py — LDA, SVM baselines + EEGNet (TensorFlow)
"""
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# LDA
def build_lda() -> Pipeline:
    return Pipeline([
        ('scaler', StandardScaler()),
        ('lda',    LinearDiscriminantAnalysis(solver='svd'))
    ])


# SVM
def build_svm(C: float = 1.0, kernel: str = 'rbf') -> Pipeline:
    return Pipeline([
        ('scaler', StandardScaler()),
        ('svm',    SVC(C=C, kernel=kernel, probability=True, class_weight='balanced'))
    ])


# EEGNet
def build_eegnet(n_channels: int, n_times: int,
                 nb_classes: int = 2,
                 F1: int = 8, D: int = 2, F2: int = 16,
                 dropout: float = 0.5):
    """
    EEGNet — Lawhern et al. 2018 (TensorFlow / Keras).
    Input shape : (batch, n_channels, n_times, 1) [channels_last]
    """
    import tensorflow as tf
    from tensorflow.keras import Model
    from tensorflow.keras.layers import (
        Input, Conv2D, DepthwiseConv2D, SeparableConv2D,
        BatchNormalization, AveragePooling2D, Dropout,
        Activation, Flatten, Dense
    )
    from tensorflow.keras.constraints import max_norm

    # 1. Update Input shape for channels_last
    inp = Input(shape=(n_channels, n_times, 1))

    # Block 1: Temporal convolution + Depthwise spatial filter
    x = Conv2D(F1, (1, 64), padding='same', use_bias=False)(inp)
    x = BatchNormalization(axis=-1)(x)
    x = DepthwiseConv2D((n_channels, 1), use_bias=False,
                        depth_multiplier=D,
                        depthwise_constraint=max_norm(1.))(x)
    x = BatchNormalization(axis=-1)(x)
    x = Activation('elu')(x)
    x = AveragePooling2D((1, 4))(x)
    x = Dropout(dropout)(x)

    # Block 2: Separable convolution
    x = SeparableConv2D(F2, (1, 16), padding='same', use_bias=False)(x)
    x = BatchNormalization(axis=-1)(x)
    x = Activation('elu')(x)
    x = AveragePooling2D((1, 8))(x)
    x = Dropout(dropout)(x)

    # Classifier head
    x   = Flatten()(x)
    out = Dense(nb_classes, activation='softmax',
                kernel_constraint=max_norm(0.25))(x)

    model = Model(inputs=inp, outputs=out)
    return model

Writing src/models.py


In [ ]:
%%writefile src/evaluate.py
"""
evaluate.py — Metrics, ITR, confusion matrix, character decoding
"""
import os, math
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score)

os.makedirs('results', exist_ok=True)

# P300 character grid
CHAR_MATRIX = np.array([
    ['A','B','C','D','E','F'],
    ['G','H','I','J','K','L'],
    ['M','N','O','P','Q','R'],
    ['S','T','U','V','W','X'],
    ['Y','Z','1','2','3','4'],
    ['5','6','7','8','9','_']
])


# Information Transfer Rate
def information_transfer_rate(N: int, P: float, T: float) -> float:
    """
    ITR in bits/minute.
    N = number of symbols (36 for 6×6 grid)
    P = classification accuracy  (0 – 1)
    T = trial duration in seconds
    """
    if P <= 0 or P >= 1:
        return 0.0
    B = (math.log2(N)
         + P * math.log2(P)
         + (1 - P) * math.log2((1 - P) / (N - 1)))
    return B * (60.0 / T)

# Classification report
def evaluate_classifier(model, X, y_true,
                         model_name: str = 'Model',
                         save_plot:  bool = True) -> dict:
    """
    Predict, print full report, save confusion matrix.
    Returns dict with accuracy, auc, itr.
    """
    y_pred = model.predict(X)
    acc    = accuracy_score(y_true, y_pred)

    try:
        probs = model.predict_proba(X)[:, 1]
        auc   = roc_auc_score(y_true, probs)
    except Exception:
        auc = float('nan')

    # Trial time: 10 repetitions × 12 flashes × ~125 ms each
    T_trial = 10 * 12 * 0.125
    itr_val = information_transfer_rate(N=36, P=acc, T=T_trial)

    print(f"\n{'='*55}")
    print(f"  {model_name}")
    print(f"  Accuracy : {acc*100:.2f}%")
    print(f"  AUC      : {auc:.4f}")
    print(f"  ITR      : {itr_val:.1f} bits/min")
    print(f"{'='*55}")
    print(classification_report(y_true, y_pred,
                                 target_names=['NonTarget','Target']))

    if save_plot:
        _plot_confusion(y_true, y_pred, model_name)

    return {'accuracy': acc, 'auc': auc, 'itr': itr_val}


def _plot_confusion(y_true, y_pred, model_name: str):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['NonTarget', 'Target'],
                yticklabels=['NonTarget', 'Target'])
    plt.title(f'{model_name} — Confusion Matrix')
    plt.ylabel('True'); plt.xlabel('Predicted')
    plt.tight_layout()
    fname = f"results/{model_name.replace(' ', '_')}_confusion.png"
    plt.savefig(fname, dpi=100)
    plt.close()
    print(f"  → Saved: {fname}")


# ERP visualisation
def plot_erp(X: np.ndarray, y: np.ndarray,
             fs: int = 256, channel_idx: int = 10,
             subject_id: int = 1):
    """
    Plot average ERP for Target vs NonTarget at one channel.
    """
    tmin, tmax = -0.1, 0.8
    times = np.linspace(tmin, tmax, X.shape[2])

    target     = X[y == 1, channel_idx, :].mean(axis=0)
    non_target = X[y == 0, channel_idx, :].mean(axis=0)

    plt.figure(figsize=(9, 4))
    plt.plot(times * 1000, target,     label='Target',     linewidth=2)
    plt.plot(times * 1000, non_target, label='Non-Target', linewidth=2,
             linestyle='--')
    plt.axvline(0,   color='k', linestyle=':', linewidth=1)
    plt.axvline(300, color='gray', linestyle=':', linewidth=1, label='~P300')
    plt.xlabel('Time (ms)'); plt.ylabel('Amplitude (µV)')
    plt.title(f'ERP — Subject {subject_id} — Channel {channel_idx}')
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
    fname = f'results/erp_subject_{subject_id:02d}.png'
    plt.savefig(fname, dpi=100); plt.close()
    print(f"  → ERP saved: {fname}")


# Cross-subject summary table
def print_summary_table(results: dict):
    """
    results: {subject_id: {'lda': {}, 'svm': {}}}
    """
    print(f"\n{'Subject':>8} {'LDA Acc':>10} {'SVM Acc':>10} "
          f"{'LDA ITR':>10} {'SVM ITR':>10}")
    print('-' * 55)
    for sid, r in results.items():
        print(f"{sid:>8} "
              f"{r['lda']['accuracy']*100:>9.1f}% "
              f"{r['svm']['accuracy']*100:>9.1f}% "
              f"{r['lda']['itr']:>9.1f}  "
              f"{r['svm']['itr']:>9.1f}")

Writing src/evaluate.py


In [ ]:
%%writefile main.py
"""
main.py — Full EEG Brain Speller Pipeline
Dataset : BNCI2014_009 (P300 speller, 10 subjects, MOABB)
"""


import os, sys, argparse
import numpy as np
import joblib
import matplotlib
matplotlib.use('Agg')
os.makedirs('results', exist_ok=True)
os.makedirs('models',  exist_ok=True)
from src.preprocess import download_dataset, load_subject, apply_notch_to_epochs, apply_ica_to_epochs
from src.features   import extract_features
from src.models     import build_lda, build_svm
from src.evaluate   import (evaluate_classifier, plot_erp,
                             print_summary_table, information_transfer_rate)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

# CLI
parser = argparse.ArgumentParser()
parser.add_argument('subjects', type=int, nargs='+',
                    default=list(range(1, 11)),
                    help='Subject IDs to process (1-10)')
parser.add_argument('method', type=str, default='downsample',
                    choices=['downsample', 'pca', 'xdawn'],
                    help='Feature extraction method')
parser.add_argument('skip-download', action='store_true')
args = parser.parse_args()

if not args.skip_download:
    download_dataset()

# Main loop
all_results = {}

for sid in args.subjects:
    print(f"\n{'#'*60}")
    print(f"  SUBJECT {sid:02d}")
    print(f"{'#'*60}")

    # 1. Load epochs
    X, y, meta = load_subject(sid)
    X = apply_notch_to_epochs(X)
    X = apply_ica_to_epochs(X)

    # 2. ERP plot
    plot_erp(X, y, subject_id=sid)

    # ── 3. Train / test split (80 / 20)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    # 4. Feature extraction
    print(f"\nFeature extraction method: {args.method}")
    X_tr, X_te, scaler, transformer = extract_features(
        X_train, y_train, X_test, method=args.method
    )

    # 5. Cross-validation on training set
    print("\n 5-Fold Cross-Validation (training set)")
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for name, clf in [('LDA', build_lda()), ('SVM', build_svm())]:
        scores = cross_val_score(clf, X_tr, y_train,
                                 cv=cv, scoring='accuracy', n_jobs=-1)
        print(f"  {name}: {scores.mean()*100:.1f}% ± {scores.std()*100:.1f}%")

    # 6. Train final models
    lda = build_lda(); lda.fit(X_tr, y_train)
    svm = build_svm(); svm.fit(X_tr, y_train)

    # 7. Test-set evaluation
    print("\n Test-set evaluation")
    r_lda = evaluate_classifier(lda, X_te, y_test, f'LDA S{sid:02d}')
    r_svm = evaluate_classifier(svm, X_te, y_test, f'SVM S{sid:02d}')
    all_results[sid] = {'lda': r_lda, 'svm': r_svm}

    # 8. Save models
    joblib.dump({'model': lda, 'scaler': scaler},
                f'models/lda_subject_{sid:02d}.pkl')
    joblib.dump({'model': svm, 'scaler': scaler},
                f'models/svm_subject_{sid:02d}.pkl')
    print(f"  Models saved → models/lda_subject_{sid:02d}.pkl")
    print(f"                 models/svm_subject_{sid:02d}.pkl")

# 9. Summary table
print(f"\n{'='*60}")
print("  FINAL SUMMARY ACROSS ALL SUBJECTS")
print(f"{'='*60}")
print_summary_table(all_results)
print("\n✅ Pipeline complete! Results saved to results/ and models/\n")

Writing main.py


In [ ]:
%%writefile train_eegnet.py
"""
train_eegnet.py — Train EEGNet on BNCI2014_009
Run: python train_eegnet.py --subject 1
"""


import os, argparse
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs('results', exist_ok=True)
os.makedirs('models',  exist_ok=True)

from src.preprocess import load_subject, apply_notch_to_epochs
from src.models     import build_eegnet
from src.evaluate   import information_transfer_rate

import tensorflow as tf
from tensorflow.keras.utils    import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection    import train_test_split
from sklearn.metrics            import accuracy_score

parser = argparse.ArgumentParser()
parser.add_argument('--subject', type=int, default=1)
parser.add_argument('--epochs',  type=int, default=100)
args = parser.parse_args()

SID = args.subject

# 1. Load data
print(f"Loading Subject {SID:02d}...")
X, y, _ = load_subject(SID)
X        = apply_notch_to_epochs(X)
X_4d = X[:, :, :, np.newaxis]
y_oh = to_categorical(y, num_classes=2)

X_train, X_val, y_train, y_val, y_tr_raw, y_va_raw = train_test_split(
    X_4d, y_oh, y,
    test_size=0.2, stratify=y, random_state=42
)

print(f"Train: {X_train.shape}  |  Val: {X_val.shape}")
print(f"Target ratio — train: {y_tr_raw.mean():.2f}  val: {y_va_raw.mean():.2f}")

_, n_channels, n_times, _ = X_train.shape

# 2. Build model
model = build_eegnet(n_channels=n_channels, n_times=n_times)
model.summary()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 3. Class weights (handle target/non-target imbalance)
n_target     = y_tr_raw.sum()
n_non_target = len(y_tr_raw) - n_target
class_weight = {0: 1.0,
                1: n_non_target / max(n_target, 1)}
print(f"\nClass weights: NonTarget=1.0, Target={class_weight[1]:.2f}")

# 4. Callbacks
ckpt_path = f'models/eegnet_subject_{SID:02d}.keras'
callbacks = [
    EarlyStopping(patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=7, verbose=1),
    ModelCheckpoint(ckpt_path, save_best_only=True, verbose=0)
]

# 5. Train
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=args.epochs,
    batch_size=64,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)

# 6. Evaluate
y_pred_prob  = model.predict(X_val)
y_pred_class = np.argmax(y_pred_prob, axis=1)
acc = accuracy_score(y_va_raw, y_pred_class)
T_trial = 10 * 12 * 0.125
itr_val = information_transfer_rate(N=36, P=acc, T=T_trial)

print(f"\n{'='*45}")
print(f"  EEGNet — Subject {SID:02d}")
print(f"  Val Accuracy : {acc*100:.2f}%")
print(f"  ITR estimate : {itr_val:.1f} bits/min")
print(f"  Model saved  : {ckpt_path}")
print(f"{'='*45}")

# 7. Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'],     label='Train', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val',   linewidth=2)
axes[0].set_title(f'EEGNet Accuracy — S{SID:02d}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'],         label='Train', linewidth=2)
axes[1].plot(history.history['val_loss'],     label='Val',   linewidth=2)
axes[1].set_title(f'EEGNet Loss — S{SID:02d}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plot_path = f'results/eegnet_training_S{SID:02d}.png'
plt.savefig(plot_path, dpi=100); plt.close()
print(f"  Training plot: {plot_path}")

Writing train_eegnet.py


In [ ]:
!python main.py

Attempting to create new mne-python configuration file:
/root/.mne/mne-python.json
Could not read the /root/.mne/mne-python.json json file during the writing. Assuming it is empty. Got: Expecting value: line 1 column 1 (char 0)
/usr/local/lib/python3.12/dist-packages/moabb/datasets/download.py:97: RuntimeWarning: Setting non-standard config type: "MNE_DATASETS_BNCI_PATH"
  set_config(key, get_config("MNE_DATA"))
/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'lampx.tugraz.at'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
100%|█████████████████████████████████████| 18.5M/18.5M [00:00<00:00, 7.86GB/s]
SHA256 hash of downloaded file: beddf78f1834ddef15553e32c9d18c46bc9b3fd244ef3a8e2fe362066dfb027d
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't c

In [ ]:
!python train_eegnet.py --subject 1

2026-04-06 21:57:49.779825: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-06 21:57:50.427006: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-06 21:57:50.866635: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775512671.438115   25777 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775512671.546699   25777 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775512672.027777   25777 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin